In [72]:
import sys
from pathlib import Path

import numpy as np
import plotly.express as px
from IPython.display import display

from data.clean_data import basic_clean
from data.load_data import load_raw_data
from eda.summary import data_summary

PROJECT_ROOT = Path.cwd().resolve().parents[0]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [73]:
df_raw = load_raw_data()
df = basic_clean(df_raw)

print("Shape:", df.shape)
display(df.head())
display(data_summary(df))

Shape: (100000, 54)


,person_id,age,sex,region,urban_rural,income,education,marital_status,employment_status,household_size,...,liver_disease,arthritis,mental_health,proc_imaging_count,proc_surgery_count,proc_physio_count,proc_consult_count,proc_lab_count,is_high_risk,had_major_procedure
0,75722,52,Female,North,Suburban,22700.0,Doctorate,Married,Retired,3,...,0,1,0,1,0,2,0,1,0,0
1,80185,79,Female,North,Urban,12800.0,No HS,Married,Employed,3,...,0,1,1,0,0,1,0,1,1,0
2,19865,68,Male,North,Rural,40700.0,HS,Married,Retired,5,...,0,0,1,1,0,2,1,0,1,0
3,76700,15,Male,North,Suburban,15600.0,Some College,Married,Self-employed,5,...,0,0,0,1,0,0,1,0,0,0
4,92992,53,Male,Central,Suburban,89600.0,Doctorate,Married,Self-employed,2,...,0,1,0,2,0,1,1,0,1,0


,dtype,missing_count,missing_pct,n_unique
alcohol_freq,str,30083,30.083,3
person_id,int64,0,0.000,100000
annual_medical_cost,float64,0,0.000,91299
total_claims_paid,float64,0,0.000,56650
annual_premium,float64,0,0.000,55538
avg_claim_amount,float64,0,0.000,53071
monthly_premium,float64,0,0.000,12339
income,float64,0,0.000,2987
ldl,float64,0,0.000,1881
hba1c,float64,0,0.000,732


In [74]:
print(df.columns.tolist())

['person_id', 'age', 'sex', 'region', 'urban_rural', 'income', 'education', 'marital_status', 'employment_status', 'household_size', 'dependents', 'bmi', 'smoker', 'alcohol_freq', 'visits_last_year', 'hospitalizations_last_3yrs', 'days_hospitalized_last_3yrs', 'medication_count', 'systolic_bp', 'diastolic_bp', 'ldl', 'hba1c', 'plan_type', 'network_tier', 'deductible', 'copay', 'policy_term_years', 'policy_changes_last_2yrs', 'provider_quality', 'risk_score', 'annual_medical_cost', 'annual_premium', 'monthly_premium', 'claims_count', 'avg_claim_amount', 'total_claims_paid', 'chronic_count', 'hypertension', 'diabetes', 'asthma', 'copd', 'cardiovascular_disease', 'cancer_history', 'kidney_disease', 'liver_disease', 'arthritis', 'mental_health', 'proc_imaging_count', 'proc_surgery_count', 'proc_physio_count', 'proc_consult_count', 'proc_lab_count', 'is_high_risk', 'had_major_procedure']


In [75]:
target = "annual_medical_cost"
log_target = "log1p_annual_medical_cost"

df[log_target] = np.log1p(df[target])

print(df[target].describe(percentiles=[0.50, 0.75, 0.90, 0.95, 0.99, 0.995]))

print("\nSkewness:", df[target].skew())
print("Kurtosis:", df[target].kurt())

mean_cost = df[target].mean()
median_cost = df[target].median()
p99_cost = df[target].quantile(0.99)

mean_log_cost = df[log_target].mean()
median_log_cost = df[log_target].median()

count    100000.000000
mean       3009.451907
std        3127.462822
min          55.550000
50%        2082.575000
75%        3707.957500
90%        6227.092000
95%        8570.350500
99%       15293.682200
99.5%     19212.947050
max       65724.900000
Name: annual_medical_cost, dtype: float64

Skewness: 4.030317894409436
Kurtosis: 32.559205950930746


In [76]:
fig = px.histogram(
    df,
    x=target,
    nbins=60,
    title="Distribution of Annual Medical Cost",
    template="plotly_white",
    opacity=0.85
)

fig.add_vline(
    x=mean_cost,
    line_dash="dash",
    annotation_text="Mean",
    annotation_position="top right"
)

fig.add_vline(
    x=median_cost,
    line_dash="dot",
    annotation_text="Median",
    annotation_position="top left"
)

fig.update_layout(
    xaxis_title="Annual Medical Cost",
    yaxis_title="Count"
)

fig.show()

In [77]:
df_body = df[df[target] <= p99_cost]

fig = px.histogram(
    df_body,
    x=target,
    nbins=60,
    title="Distribution of Annual Medical Cost (Up to 99th Percentile)",
    template="plotly_white",
    opacity=0.85
)

fig.add_vline(
    x=df_body[target].mean(),
    line_dash="dash",
    annotation_text="Mean",
    annotation_position="top right"
)

fig.add_vline(
    x=df_body[target].median(),
    line_dash="dot",
    annotation_text="Median",
    annotation_position="top left"
)

fig.update_layout(
    xaxis_title="Annual Medical Cost",
    yaxis_title="Count"
)

fig.show()

In [78]:
fig = px.histogram(
    df,
    x=log_target,
    nbins=60,
    title="Distribution of log(1 + Annual Medical Cost)",
    template="plotly_white",
    opacity=0.85
)

fig.add_vline(
    x=mean_log_cost,
    line_dash="dash",
    annotation_text="Mean",
    annotation_position="top right"
)

fig.add_vline(
    x=median_log_cost,
    line_dash="dot",
    annotation_text="Median",
    annotation_position="top left"
)

fig.update_layout(
    xaxis_title="log(1 + Annual Medical Cost)",
    yaxis_title="Count"
)

fig.show()

In [79]:
fig = px.box(
    df,
    x=log_target,
    points="outliers",
    title="Boxplot of log(1 + Annual Medical Cost)",
    template="plotly_white"
)

fig.update_layout(
    xaxis_title="log(1 + Annual Medical Cost)"
)

fig.show()

In [80]:
df_plot = df.sample(5000, random_state=42).copy()

rng = np.random.default_rng(42)
df_plot["age_jitter"] = df_plot["age"] + rng.uniform(-0.25, 0.25, size=len(df_plot))

fig = px.scatter(
    df_plot,
    x="age_jitter",
    y=log_target,
    color="smoker",
    hover_data=["age", "bmi", "chronic_count", "hospitalizations_last_3yrs"],
    title="log(1 + Annual Medical Cost) vs Age",
    template="plotly_white"
)

fig.update_traces(marker=dict(size=5), opacity=0.35)

fig.update_layout(
    xaxis_title="Age",
    yaxis_title="log(1 + Annual Medical Cost)"
)

fig.show()

In [81]:
fig = px.box(
    df,
    x="hospitalizations_last_3yrs",
    y=log_target,
    points="outliers",
    title="log(1 + Annual Medical Cost) by Hospitalisations in Last 3 Years",
    template="plotly_white"
)

fig.update_layout(
    xaxis_title="Hospitalisations in Last 3 Years",
    yaxis_title="log(1 + Annual Medical Cost)"
)

fig.show()

In [82]:
fig = px.box(
    df,
    x="chronic_count",
    y=log_target,
    points="outliers",
    title="log(1 + Annual Medical Cost) by Chronic Count",
    template="plotly_white"
)

fig.update_layout(
    xaxis_title="Chronic Count",
    yaxis_title="log(1 + Annual Medical Cost)"
)

fig.show()

In [83]:
print(df["hospitalizations_last_3yrs"].value_counts().sort_index())
print()
print(df["chronic_count"].value_counts().sort_index())

hospitalizations_last_3yrs
0    91031
1     8582
2      379
3        8
Name: count, dtype: int64

chronic_count
0    46532
1    37579
2    13111
3     2452
4      316
5        9
6        1
Name: count, dtype: int64


In [84]:
df["hospitalizations_grouped"] = df["hospitalizations_last_3yrs"].replace({
    2: "2+",
    3: "2+"
}).astype(str)

df["chronic_count_grouped"] = df["chronic_count"].apply(
    lambda x: str(x) if x <= 3 else "4+"
)

print(df["hospitalizations_grouped"].value_counts().sort_index())
print()
print(df["chronic_count_grouped"].value_counts().sort_index())

hospitalizations_grouped
0     91031
1      8582
2+      387
Name: count, dtype: int64

chronic_count_grouped
0     46532
1     37579
2     13111
3      2452
4+      326
Name: count, dtype: int64
